In [215]:
from typing import List


# Representa um número binário com tamanho fixo
class Bin:
    def __init__(self, value: int, size: int):
        self.value = value  # Valor inteiro
        self.size = size  # Tamanho em bits
        
    # Representação para debug (em hexadecimal)
    def __repr__(self):
        return f"Bin(value={self}, size={self.size})"

    # Representação em string (em hexadecimal)
    def __str__(self, type="x", group_size=0):
        hex_digits = (self.size + 3) // 4
        return f"0x{self.value:0{hex_digits}X}"

    # Format_spec example: "x", "b", or "b_4" (for grouping)
    def __format__(self, format_spec):
        if not format_spec:
            return str(self)
        parts = format_spec.split("_")
        fmt = parts[0]
        group_size = int(parts[1]) if len(parts) > 1 else 0

        if fmt in ("b", "B", "x", "X"):
            return self.to_str(fmt, group_size)
        else:
            return str(self)

    # Verifica igualdade com outro objeto Bin
    def __eq__(self, other):
        if isinstance(other, Bin):
            return self.value == other.value and self.size == other.size
        return False

    def __xor__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value ^ other.value, self.size)

    def __xor__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value ^ other.value, self.size)

    def __or__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value | other.value, self.size)

    def __and__(self, other: "Bin") -> "Bin":
        self.assert_same_size(other)
        return Bin(self.value & other.value, self.size)


    def __mul__(self, other: "Bin") -> "Bin":
        result_value = self.value * other.value
        result_size = result_value.bit_length() or 1
        return Bin(result_value, result_size)


    def __add__(self, other: "Bin") -> "Bin":
        result_value = self.value + other.value
        result_size = result_value.bit_length() or 1
        return Bin(result_value, result_size)


    def __lshift__(self, shift_value: int) -> "Bin":
        shift_value %= self.size
        discarded = self.value >> (self.size - shift_value)
        mask = (1 << self.size) - 1
        shifted = (self.value << shift_value) & mask
        result = shifted | discarded
        return Bin(result, self.size)


    def __rshift__(self, shift_value: int) -> "Bin":
        shift_value %= self.size
        discarded = (self.value << (self.size - shift_value)) & ((1 << self.size) - 1)
        shifted = self.value >> shift_value
        result = shifted | discarded
        return Bin(result, self.size)


    def copy(self):
        return Bin(self.value, self.size)

    def to_list(self) -> List[int]:
        return [(self.value >> i) & 1 for i in reversed(range(self.size))]


    
    @staticmethod
    def finite_field_mul(a: "Bin", b: "Bin", modulus: int = 0x11B) -> "Bin":
        assert a.size == b.size, "Operands must be the same size"
        result = 0
        a_val = a.value
        b_val = b.value
        for _ in range(8):  # Since we're working with GF(2^8)
            if b_val & 1:
                result ^= a_val
            b_val >>= 1
            a_val <<= 1
            if a_val & 0x100:  # Check if x^8 is set (overflow)
                a_val ^= modulus  # Modular reduction

        return Bin(result, a.size)



    def to_str(self, fmt="x", group_size=0):
        if fmt in ("x", "X"):
            hex_digits = (self.size + 3) // 4
            raw_hex = f"{self.value:0{hex_digits}{fmt}}"
            if group_size <= 0:
                return f"0x{raw_hex}"
            else:
                # Group hex digits from right to left (common style)
                groups = []
                for i in range(len(raw_hex), 0, -group_size):
                    start = max(0, i - group_size)
                    groups.append(raw_hex[start:i])
                grouped = "_".join(reversed(groups))
                return f"0x{grouped}"

        elif fmt in ("b", "B"):
            raw_bin = f"{self.value:0{self.size}b}"
            if group_size <= 0:
                return f"0b{raw_bin}"
            else:
                grouped = "_".join(
                    raw_bin[i : i + group_size]
                    for i in range(0, len(raw_bin), group_size)
                )
                return f"0b{grouped}"
        else:
            raise ValueError(f"Unsupported format: {fmt}")

    # Realiza operação XOR bit a bit entre vários Bin
    def xor(self, *bins: "Bin"):
        for bin in bins:
            self.assert_same_size(bin)
            self.value ^= bin.value
        return self

    # Extrai bits com base em uma tabela de posições e retorna como novo Bin
    def extract(self, table: List[List[int]]):
        extracted_bits = []
        for row in table:
            for bit_pos in row:
                bit = (self.value >> (self.size - bit_pos)) & 1
                extracted_bits.append(bit)
        result = 0
        for bit in extracted_bits:
            result = (result << 1) | bit
        return Bin(result, len(extracted_bits))

    # Divide o valor Bin em duas metades e retorna como dois objetos Bin
    def halve(self):
        half = (self.size + 1) // 2
        left = (self.value >> half) & ((1 << half) - 1)
        right = self.value & ((1 << half) - 1)
        return Bin(left, half), Bin(right, half)

    # Divide o Bin em pedaços menores de tamanho fixo
    def split(self, chunk_size: int):
        remainder = self.size % chunk_size
        if remainder != 0:
            # Calculate how many bits to pad
            pad_size = chunk_size - remainder
            self.value <<= pad_size  # Shift left to pad with zeros
            self.size += pad_size

        chunks: List[Bin] = []
        for i in range(0, self.size, chunk_size):
            chunk_value = (self.value >> (self.size - i - chunk_size)) & (
                (1 << chunk_size) - 1
            )
            chunks.append(Bin(chunk_value, chunk_size))
        return chunks

    # Troca as metades esquerda e direita de um Bin
    def swap(self):
        left, right = self.halve()
        right.extend(left)
        self.value = right.value

    # Anexa os bits de outro Bin ao final deste Bin
    def extend(self, bin: "Bin"):
        self.value = (self.value << bin.size) | bin.value
        self.size += bin.size

    def assert_same_size(self, bin: "Bin"):
        if self.size != bin.size:
            raise ValueError("Bin sizes must match for bitwise operations")

    @staticmethod
    # Junta vários objetos Bin em um só
    def fuse(*bins: "Bin"):
        total_value = 0
        total_size = 0
        for b in bins:
            total_value = (total_value << b.size) | b.value
            total_size += b.size
        return Bin(total_value, total_size)

    # Converte uma str para um bin
    @staticmethod
    def from_hex(hex_str: str):
        return Bin(int(hex_str, 16), len(hex_str) * 4)

In [216]:
class BinBlock:
    block: List[List[Bin]] = []

    def __init__(self, bin=Bin(0x0, 128)):
        if bin.size != 128:
            raise ValueError("BinBlock requires a 128-bit Bin")  # TODO: 192, 256

        block = [[], [], [], []] #TODO: class Matrix?
        bytes = bin.split(8)
        for i in range(0, 16, 4):
            for j in range(4):
                block[j].append(bytes[i + j])

        self.block = block
        self.bytes = 16

    def __repr__(self):
        rows = ",\n  ".join([str(row) for row in self.block])
        return f"BinBlock([\n  {rows}\n])"

    def __str__(self):
        rows = "\n".join([" ".join([str(b) for b in row]) for row in self.block])
        return rows

    def __eq__(self, other):
        if not isinstance(other, BinBlock):
            return False
        return self.block == other.block and self.bytes == other.bytes

    def __xor__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a ^ b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.bytes = self.bytes
        return result

    def __or__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a | b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.bytes = self.bytes
        return result

    def __and__(self, other: "BinBlock"):
        if not isinstance(other, BinBlock):
            raise TypeError("Operand must be a BinBlock")

        result_block = []
        for row_self, row_other in zip(self.block, other.block):
            result_row = [a & b for a, b in zip(row_self, row_other)]
            result_block.append(result_row)

        result = BinBlock.__new__(BinBlock)
        result.block = result_block
        result.bytes = self.bytes
        return result

    def shift_line(self, l:int, amt:int):
        line = Bin.fuse(*self.block[l])
        line = line << amt
        self.block[l] = line.split(8)

    def get_column(self, c: int) -> List[Bin]:
        if not (0 <= c < 4):
            raise IndexError("Column index must be between 0 and 3")
        return [self.block[row][c] for row in range(4)]


    def get_columns(self) -> List[List[Bin]]:
        return [[self.block[row][c] for row in range(4)] for c in range(4)]


    def copy(self):
        new_block = BinBlock()
        new_block.block = self.block.copy()
        return new_block

    def to_bin(self) -> Bin:
        # Flatten row-wise and fuse into one Bin
        flat = [b for row in self.block for b in row]
        return Bin.fuse(*flat)

In [217]:
r_con = [ 
    0x00, 0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40,
    0x80, 0x1B, 0x36, 0x6C, 0xD8, 0xAB, 0x4D, 0x9A,
    0x2F, 0x5E, 0xBC, 0x63, 0xC6, 0x97, 0x35, 0x6A,
    0xD4, 0xB3, 0x7D, 0xFA, 0xEF, 0xC5, 0x91, 0x39,
]

mix_columns_table = [
    [2, 3, 1, 1],
    [1, 2, 3, 1],
    [1, 1, 2, 3],
    [3, 1, 1, 2]
]

substitution_box = [
    [0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76],
    [0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0],
    [0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15],
    [0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75],
    [0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84],
    [0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF],
    [0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8],
    [0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2],
    [0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73],
    [0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB],
    [0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79],
    [0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08],
    [0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A],
    [0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E],
    [0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF],
    [0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16],
]

sbox_inv = [
    [0x52, 0x09, 0x6A, 0xD5, 0x30, 0x36, 0xA5, 0x38, 0xBF, 0x40, 0xA3, 0x9E, 0x81, 0xF3, 0xD7, 0xFB],
    [0x7C, 0xE3, 0x39, 0x82, 0x9B, 0x2F, 0xFF, 0x87, 0x34, 0x8E, 0x43, 0x44, 0xC4, 0xDE, 0xE9, 0xCB],
    [0x54, 0x7B, 0x94, 0x32, 0xA6, 0xC2, 0x23, 0x3D, 0xEE, 0x4C, 0x95, 0x0B, 0x42, 0xFA, 0xC3, 0x4E],
    [0x08, 0x2E, 0xA1, 0x66, 0x28, 0xD9, 0x24, 0xB2, 0x76, 0x5B, 0xA2, 0x49, 0x6D, 0x8B, 0xD1, 0x25],
    [0x72, 0xF8, 0xF6, 0x64, 0x86, 0x68, 0x98, 0x16, 0xD4, 0xA4, 0x5C, 0xCC, 0x5D, 0x65, 0xB6, 0x92],
    [0x6C, 0x70, 0x48, 0x50, 0xFD, 0xED, 0xB9, 0xDA, 0x5E, 0x15, 0x46, 0x57, 0xA7, 0x8D, 0x9D, 0x84],
    [0x90, 0xD8, 0xAB, 0x00, 0x8C, 0xBC, 0xD3, 0x0A, 0xF7, 0xE4, 0x58, 0x05, 0xB8, 0xB3, 0x45, 0x06],
    [0xD0, 0x2C, 0x1E, 0x8F, 0xCA, 0x3F, 0x0F, 0x02, 0xC1, 0xAF, 0xBD, 0x03, 0x01, 0x13, 0x8A, 0x6B],
    [0x3A, 0x91, 0x11, 0x41, 0x4F, 0x67, 0xDC, 0xEA, 0x97, 0xF2, 0xCF, 0xCE, 0xF0, 0xB4, 0xE6, 0x73],
    [0x96, 0xAC, 0x74, 0x22, 0xE7, 0xAD, 0x35, 0x85, 0xE2, 0xF9, 0x37, 0xE8, 0x1C, 0x75, 0xDF, 0x6E],
    [0x47, 0xF1, 0x1A, 0x71, 0x1D, 0x29, 0xC5, 0x89, 0x6F, 0xB7, 0x62, 0x0E, 0xAA, 0x18, 0xBE, 0x1B],
    [0xFC, 0x56, 0x3E, 0x4B, 0xC6, 0xD2, 0x79, 0x20, 0x9A, 0xDB, 0xC0, 0xFE, 0x78, 0xCD, 0x5A, 0xF4],
    [0x1F, 0xDD, 0xA8, 0x33, 0x88, 0x07, 0xC7, 0x31, 0xB1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xEC, 0x5F],
    [0x60, 0x51, 0x7F, 0xA9, 0x19, 0xB5, 0x4A, 0x0D, 0x2D, 0xE5, 0x7A, 0x9F, 0x93, 0xC9, 0x9C, 0xEF],
    [0xA0, 0xE0, 0x3B, 0x4D, 0xAE, 0x2A, 0xF5, 0xB0, 0xC8, 0xEB, 0xBB, 0x3C, 0x83, 0x53, 0x99, 0x61],
    [0x17, 0x2B, 0x04, 0x7E, 0xBA, 0x77, 0xD6, 0x26, 0xE1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0C, 0x7D],
]

In [218]:
class Logger:
    def __init__(self):
        self.logs = {}

    def log(self, description: str, obj, title: str = "default"):
        if title not in self.logs:
            self.logs[title] = []
        self.logs[title].append((description, obj))

    def show(self, title: str = None):
        if title:
            logs = self.logs.get(title, [])
            print(f"=== Logs for phase '{title}' ===\n")
            for i, (desc, obj) in enumerate(logs, 1):
                print(f"[{i}] {desc}:\n{str(obj)}\n")
        else:
            for title, entries in self.logs.items():
                print(f"=== Logs for phase '{title}' ===\n")
                for i, (desc, obj) in enumerate(entries, 1):
                    print(f"[{i}] {desc}:\n{str(obj)}\n")

In [219]:
class AES:

    def __init__(self, key_size=128, mode="CBC"):
        if key_size not in [128, 192, 256]:
            raise ValueError("Invalid key size for AES")

        round_numbers = {128: 10, 192: 12, 256: 14}

        self.logger = Logger()
        self.key_size = key_size
        self.rounds = round_numbers[key_size]
        self.mode = mode

    @staticmethod
    def sbox(byte: Bin):
        if byte.size != 8:
            raise ValueError("Argument must be a byte")

        l, c = [b.value for b in byte.halve()]
        return Bin(substitution_box[l][c], 8)

    def key_expansion(self, key: Bin):
        def g(w3: Bin, r: int):
            rotw = w3 << 8
            bytes = rotw.split(8)
            bytes = list(map(AES.sbox, bytes))

            subw = Bin.fuse(*bytes)
            rcon = Bin(r_con[r + 1], 8)
            rcon.extend(Bin(0, 24))
            g_result = subw ^ rcon

            # self.logger.log(f"g(w3, r={r})", g_result, title="Key Expansion")
            return g_result

        key_block = BinBlock(key)
        words: List[List[Bin]] = [
            [Bin.fuse(*[key_block.block[j][i] for j in range(4)]) for i in range(4)]
        ]

        round_keys = [BinBlock(Bin.fuse(*words[0]))]
        self.logger.log(f"Round Key 0", round_keys[0], title="Key Expansion")
        for i in range(10):
            w0, w1, w2, w3 = words[i]
            w4 = w0 ^ g(w3, i)
            w5 = w4 ^ w1
            w6 = w5 ^ w2
            w7 = w6 ^ w3

            words.append([w4, w5, w6, w7])
            subkey = BinBlock(Bin.fuse(w4, w5, w6, w7))
            round_keys.append(subkey)
            self.logger.log(f"Round Key {i + 1}", subkey, title="Key Expansion")
        return round_keys

    @staticmethod
    def sub_bytes(plt: BinBlock) -> BinBlock:
        block = plt.block
        cipher_block = BinBlock()
        for i in range(4):
            for j in range(4):
                s = AES.sbox(block[i][j])
                cipher_block.block[i][j] = s

        return cipher_block

    @staticmethod
    def shift_rows(plt: BinBlock):
        cipher_block = plt.copy()
        cipher_block.shift_line(1, 8)
        cipher_block.shift_line(2, 16)
        cipher_block.shift_line(3, 24)
        return cipher_block

    @staticmethod
    def mix_columns(plt: BinBlock) -> BinBlock:
        r = Bin(0, 0)
        mct = [[Bin(k, 8) for k in l] for l in mix_columns_table]

        for c in plt.get_columns():
            for l in mct:
                byte = []
                for i in range(4):
                    byte.append(Bin.finite_field_mul(l[i], c[i]))
                r.extend(Bin.xor(*byte))

        return BinBlock(r)


    def transform(self, plt: BinBlock, subkey: BinBlock, round: int):
        last_round = round == self.rounds
        title = f"round {round}"
        self.logger.log("Round key", subkey, title=title)
        self.logger.log("Before Transformations", plt, title=title)

        block = AES.sub_bytes(plt)
        self.logger.log("After SubBytes", block, title=title)

        block = AES.shift_rows(block)
        self.logger.log("After ShiftRows", block, title=title)

        if not last_round:
            block = AES.mix_columns(block)
            self.logger.log("After MixColumns", block, title=title)

        block = block ^ subkey
        self.logger.log("After AddRoundKey", block, title=title)
        return block

    def encrypt(self, plaintext: Bin, key: Bin) -> Bin:
        self.check_key_size(key)
        plt_block = BinBlock(plaintext)
        rnd_keys = self.key_expansion(key)

        # Round 0: AddRoundKey
        cipher_block = plt_block ^ rnd_keys[0]

        self.logger.log("Plaintext block", plt_block, title="round 0")
        self.logger.log("Initial round key", rnd_keys[0], title="round 0")
        self.logger.log("After AddRoundKey", cipher_block, title="round 0")

        # Main rounds
        for i in range(1, self.rounds + 1):
            cipher_block = self.transform(cipher_block, rnd_keys[i], i)

        return cipher_block.to_bin()

    def decrypt(self, ciphertext: Bin, key: Bin) -> Bin:
        self.check_key_size(key)
        pass

    def check_key_size(self, key: Bin):
        if key.size != self.key_size:
            raise ValueError(
                f"Key does not match the expected size for AES-{self.key_size}"
            )

In [220]:
aes = AES()


# secretkeysecretk
example_key = Bin(0x73_65_63_72_65_74_6B_65_79_73_65_63_72_65_74_6B, 128)
# secretmessagenow
example_plt = Bin(0x73_65_63_72_65_74_6D_65_73_73_61_67_65_6E_6F_77, 128)


ciphertext = aes.encrypt(example_plt, example_key)
aes.logger.show("round 1")
print(ciphertext)

=== Logs for phase 'round 1' ===

[1] Round key:
0x3F 0x5A 0x23 0x51
0xF7 0x83 0xF0 0x95
0x1C 0x77 0x12 0x66
0x32 0x57 0x34 0x5F

[2] Before Transformations:
0x00 0x00 0x0A 0x17
0x00 0x00 0x00 0x0B
0x00 0x06 0x04 0x1B
0x00 0x00 0x04 0x1C

[3] After SubBytes:
0x63 0x63 0x67 0xF0
0x63 0x63 0x63 0x2B
0x63 0x6F 0xF2 0xAF
0x63 0x63 0xF2 0x9C

[4] After ShiftRows:
0x63 0x63 0x67 0xF0
0x63 0x63 0x2B 0x63
0xF2 0xAF 0x63 0x6F
0x9C 0x63 0x63 0xF2

[5] After MixColumns:
0x0D 0xAF 0xB3 0xC3
0x34 0x2C 0xF7 0x75
0x40 0xE0 0x2F 0x40
0x17 0xAF 0x27 0xF8

[6] After AddRoundKey:
0x32 0xF5 0x90 0x92
0xC3 0xAF 0x07 0xE0
0x5C 0x97 0x3D 0x26
0x25 0xF8 0x13 0xA7

0xB39E3A866343B389C63EB1B023BCEF9B


In [221]:
sk = BinBlock(Bin(0xac7766f319fadc2128d12941575c006a,128))
b = BinBlock(Bin(0xea835cf00445332d655d98ad8596b0c5,128))
aes = AES()
r = aes.transform(b,sk,1)
print(r)

0xEB 0x59 0x8B 0x1B
0x40 0x2E 0xA1 0xC3
0xF2 0x38 0x13 0x42
0x1E 0x84 0xE7 0xD6


In [222]:
b = BinBlock(Bin(0x876e46a6f24ce78c4d904ad897ecc395,128))
r = AES.mix_columns(b)
e = BinBlock(Bin(0x473794ed40d4e4a5a3703aa64c9f42bc, 128))
print(r==e)
print()
print(r)
print()
print(e)


True

0x47 0x40 0xA3 0x4C
0x37 0xD4 0x70 0x9F
0x94 0xE4 0x3A 0x42
0xED 0xA5 0xA6 0xBC

0x47 0x40 0xA3 0x4C
0x37 0xD4 0x70 0x9F
0x94 0xE4 0x3A 0x42
0xED 0xA5 0xA6 0xBC
